# Lesson 19 — อ่าน Lance โดยไม่ผ่าน LanceDB

`.lance` directory คือสัญญา LanceDB เป็นแค่ client หนึ่งตัว
บทนี้อ่านตารางเดิม 11 โพสต์ สามทาง ไม่มี `tbl.search()` สักบรรทัด
Polars · DuckDB บน Arrow · แล้วก็อ่าน directory ตรง ๆ ด้วย `pylance` ไม่ import lancedb เลย
ทุกทางต้องได้ตัวเลขเดียวกัน memory 5 · agents 3 · hardware 3

In [1]:
%pip install -q lancedb pandas polars duckdb pylance

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, urllib.request, pathlib
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

import pandas as pd
import lancedb
db = lancedb.connect("./data")
tbl = db.create_table("posts", data=load("nat_posts.jsonl"), mode="overwrite")
tbl.to_pandas()[["id", "topic", "date", "text"]].assign(text=lambda d: d.text.str[:40]).head(4)

[2026-09-10T11:49:16Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/19-duckdb-polars/data/posts.lance, it will be created


,id,topic,date,text
0,p01,memory,2026-08-28,ยอมกลับมาทำ Memory เพราะซาบซึ้งว่ามันต้อ
1,p02,memory,2026-08-20,หนังสือ... บันทึกการ... ค่อยๆสร้าง Vecto
2,p03,memory,2026-08-15,ความทรงจำระหว่างบรรทัดของ Human และ Clau
3,p04,memory,2026-07-31,ทำอินเด็กซ์ Embedding เข้าสู่ Vector Spa


**(a) Polars** — `to_polars()` มีให้ตรง ๆ ใน 0.38 ได้ LazyFrame
ต่อ `.filter` `.group_by` ของ Polars ได้เลย ตารางข้างล่างคือจำนวนโพสต์ต่อหัวข้อกับวันแรก

In [3]:
import polars as pl
via_polars = tbl.to_polars().group_by("topic").agg(pl.len().alias("n"), pl.col("date").min().alias("first_post")).sort("topic").collect()
via_polars

topic,n,first_post
str,u32,str
"""agents""",3,"""2026-05-20"""
"""hardware""",3,"""2026-05-30"""
"""memory""",5,"""2026-06-22"""


**(b) DuckDB บน Arrow** — เหมือนบทที่ 5 `to_arrow()` แล้วตั้งชื่อใน SQL ได้ทันที
ตัวเลขเดียวกับ Polars

In [4]:
import duckdb
posts = tbl.to_arrow()
via_duckdb = duckdb.sql("SELECT topic, count(*) n, min(date) first_post FROM posts GROUP BY topic ORDER BY topic").df()
via_duckdb

,topic,n,first_post
0,agents,3,2026-05-20
1,hardware,3,2026-05-30
2,memory,5,2026-06-22


**(c) ไม่มี lancedb เลย** — DuckDB มี community extension `lance` อ่าน directory ได้ตรง
แต่ยังไม่มี build ให้ทุก platform ลองก่อน ถ้าไม่มีก็ใช้ `pylance` (`import lance`)
ซึ่งคือ Rust core ตัวเดียวกับที่ LanceDB ใช้ข้างใน
ตารางบอกว่าทางไหนใช้ได้บนเครื่องนี้ กับ error บรรทัดแรกถ้าไม่ได้

In [5]:
attempts = []
try:
    duckdb.sql("INSTALL lance FROM community; LOAD lance;")
    ext = duckdb.sql("SELECT topic, count(*) n FROM lance_scan('data/posts.lance') GROUP BY topic ORDER BY topic").df()
    attempts.append({"reader": "duckdb lance extension", "works here?": "✓", "note": f"{len(ext)} topics"})
except Exception as e:
    attempts.append({"reader": "duckdb lance extension", "works here?": "✗", "note": str(e).splitlines()[0][:80]})

import lance
ds = lance.dataset("data/posts.lance")
attempts.append({"reader": "pylance lance.dataset()", "works here?": "✓", "note": f"version {ds.version}, {ds.count_rows()} rows"})
pd.DataFrame(attempts)

,reader,works here?,note
0,duckdb lance extension,✗,"HTTP Error: Failed to download extension ""lanc..."
1,pylance lance.dataset(),✓,"version 1, 11 rows"


`pylance` อ่าน directory เดียวกัน filter pushdown ทำงาน ขอเฉพาะ column ที่ต้องการได้
ผลคือ hardware 3 แถว p09 p10 p11

In [6]:
via_lance = ds.to_table(columns=["id", "topic", "date", "text"], filter="topic = 'hardware'").to_pandas()
via_lance.assign(text=lambda d: d.text.str[:40])

,id,topic,date,text
0,p09,hardware,2026-05-31,ทำเฟิร์มแวร์เอา Claude Code มาออกจอเลยคร
1,p10,hardware,2026-05-30,เอาจอ มาต่อ Claude Code BLE Bridge ลองแล
2,p11,hardware,2026-06-28,เตรียมตัวแปลงร่างกันครับ! รอบทความแผ้บบบ


**สรุปสามทาง** — ทุกทางเห็นข้อมูลเดียวกัน ต่างกันที่ต้อง import อะไร
`lance.dataset` เห็นสิ่งเดียวกับ `lancedb.open_table` เพราะอ่านไฟล์เดียวกัน
version · fragment · filter pushdown ครบ ที่ไม่มีคือ embedding registry กับ hybrid search นั่นคือของ LanceDB ชั้นบน

ทำไมสำคัญ ตาราง 835 MB ของ `session-dream` ไม่ต้องรอ MCP server ตื่น
DuckDB หรือ Polars เปิดอ่าน วิเคราะห์ export ได้เลย โดยไม่แตะโค้ดที่เขียนมัน

In [7]:
pd.DataFrame([
    {"method": "tbl.to_polars()", "needs lancedb?": "yes (to open)", "rows seen": via_polars["n"].sum(), "filter pushdown?": "lazy — yes", "vector search?": "no"},
    {"method": "duckdb over tbl.to_arrow()", "needs lancedb?": "yes (to open)", "rows seen": int(via_duckdb["n"].sum()), "filter pushdown?": "no (Arrow in memory)", "vector search?": "no"},
    {"method": "lance.dataset(path)", "needs lancedb?": "no", "rows seen": ds.count_rows(), "filter pushdown?": "yes", "vector search?": "raw only, no registry"},
    {"method": "lancedb.open_table(path)", "needs lancedb?": "yes", "rows seen": tbl.count_rows(), "filter pushdown?": "yes", "vector search?": "yes + FTS + hybrid"},
])

,method,needs lancedb?,rows seen,filter pushdown?,vector search?
0,tbl.to_polars(),yes (to open),11,lazy — yes,no
1,duckdb over tbl.to_arrow(),yes (to open),11,no (Arrow in memory),no
2,lance.dataset(path),no,11,yes,"raw only, no registry"
3,lancedb.open_table(path),yes,11,yes,yes + FTS + hybrid


ไฟล์ที่ทุกทางอ่าน คือ fragment เดียวกันใน `data/`

In [8]:
pd.DataFrame([{"fragment file": p.name[:12] + "…", "bytes": p.stat().st_size,
               "seen by lancedb": "✓", "seen by pylance": "✓" if len(ds.get_fragments()) else ""}
              for p in sorted(pathlib.Path("data/posts.lance/data").glob("*.lance"))])

,fragment file,bytes,seen by lancedb,seen by pylance
0,111001010001…,4871,✓,✓
